# 05 — KNN Query Prototype

Interactive prototype for querying the trained KNN model. Used to validate recommendation quality before building the Streamlit app.

**What it does:**
- Loads the trained model and normalised feature matrix from `data/model/`
- Provides an `search_artist()` helper to find albums by artist name
- Provides a `recommend()` function that returns the N most similar albums to a given album, excluding other albums by the same artist
- Used for manual spot-checking of results (e.g. querying "Massive Attack" to verify recommendations make sense)

**Inputs:** `data/model/knn_model.joblib`, `data/model/X_knn_norm.npz`, `data/model/album_ids_annotated.npy`, `data/model/has_features.npy`, `data/mb_album_artists.parquet`

**Outputs:** None — query and display only. The recommendation logic here is the direct prototype for `3-model/app.py`.

**Run after:** `04-knn-training.ipynb`

## Imports

Only `pandas` and `numpy` are needed at this point — the model-specific imports (`joblib`, `scipy.sparse`) are deferred to the cell that loads the model artefacts. Keeping imports close to their first use makes it easier to see which cells have external dependencies.

In [ ]:
import pandas as pd
import numpy as np

## Load album-artist lookup table

`mb_album_artists.parquet` is a join of MusicBrainz album and artist tables. It can contain multiple rows per album when an album has multiple credited artists. `.drop_duplicates(subset='album_id')` keeps only the first row per album so the table can be safely used as a direct index-lookup (one row per `album_id`). If duplicates were left in, `.loc[album_id]` would return a DataFrame instead of a Series, breaking the `recommend` function downstream.

This table serves two purposes: (1) displaying human-readable album and artist names in results, and (2) identifying the input artist so their other albums can be excluded from recommendations.

In [ ]:
lookup = (
    pd.read_parquet('../data/mb_album_artists.parquet', columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id')
    .set_index('album_id')
)

print(f"Lookup table: {len(lookup):,} albums")
print(f"Missing artist_name: {lookup['artist_name'].isna().sum():,}")
print(f"Missing album_name : {lookup['album_name'].isna().sum():,}")
lookup.head()

## search_artist helper function

A convenience function for interactive exploration in this notebook. It does a case-insensitive partial string match on `artist_name`, which is more forgiving than an exact lookup (e.g. "massive" matches "Massive Attack"). The returned DataFrame includes the `album_id` index, making it easy to copy an `album_id` into the `recommend()` call below.

This function is not used in the production app — the Streamlit selectbox handles album selection there — but it is essential for manual spot-checking during prototyping.

In [ ]:
def search_artist(name, max_results=10):
    """Case-insensitive partial match on artist_name. Returns album_id, album_name, artist_name."""
    mask = lookup['artist_name'].str.contains(name, case=False, na=False)
    return lookup[mask].head(max_results)

search_artist("Massive Attack")

## Load model artefacts

Loads the four files saved by notebook 04:

- `knn_model.joblib` — the fitted `NearestNeighbors` object used to run queries.
- `X_knn_norm.npz` — the normalised feature matrix; individual rows are sliced out as query vectors.
- `album_ids_annotated.npy` — maps model row indices back to MusicBrainz album UUIDs.
- `has_features.npy` — not used in this notebook directly, but loaded here to confirm the artefact is intact.

`album_id_to_row` inverts `album_ids_annotated` into a dictionary for O(1) lookup: given an album UUID, find its row in the model matrix. Without this, every query would require a linear scan through the array.

In [ ]:
import joblib
from scipy.sparse import load_npz

model                = joblib.load('../data/model/knn_model.joblib')
X_knn_norm           = load_npz('../data/model/X_knn_norm.npz')
album_ids_annotated  = np.load('../data/model/album_ids_annotated.npy', allow_pickle=True)
has_features         = np.load('../data/model/has_features.npy')

# Index for O(1) album_id → row lookup in the fitted model
album_id_to_row = {aid: i for i, aid in enumerate(album_ids_annotated)}

print(f"Model loaded: {X_knn_norm.shape[0]:,} albums x {X_knn_norm.shape[1]:,} features")

## recommend function

The core recommendation logic. This function is the direct prototype for `3-model/app.py` — the production Streamlit app uses essentially the same code.

**Why `n_neighbors=n*5`?** The model has no awareness of artist identity. After excluding the query album itself and all other albums by the same artist, the number of usable candidates could be significantly smaller than `n`. Fetching `n*5` candidates provides a buffer to absorb those exclusions. For an artist with 4 albums in the index, fetching 50 candidates guarantees we can still return 10 recommendations even after filtering. If an artist has an unusually large catalogue relative to `n*5`, the function may return fewer than `n` results — this is acceptable behaviour at the prototype stage.

**Same-artist exclusion logic.** The function uses string equality on `artist_name` from the lookup table. This is a simple heuristic: it will miss cases where the same artist appears under slightly different spellings, but it avoids requiring a full artist-ID join, which would need additional data loading. The production app follows the same approach.

**Early return for missing albums.** If `album_id` is not in `album_id_to_row`, the album was excluded during training (no feature data). The function returns a plain string message rather than raising an exception, making it safe to call from a UI context.

In [ ]:
def recommend(album_id, n=10):
    """
    Return the n most similar albums to album_id, excluding all albums by the
    same artist. Returns a DataFrame with album_name and artist_name, or a
    message if the album has no features.
    """
    if album_id not in album_id_to_row:
        return f"No recommendations: album {album_id} has no feature data."

    # Identify the input artist so we can exclude their other albums
    input_artist = lookup.loc[album_id, 'artist_name'] if album_id in lookup.index else None

    # Fetch extra candidates to absorb same-artist exclusions
    row = album_id_to_row[album_id]
    distances, indices = model.kneighbors(X_knn_norm[row], n_neighbors=n * 5)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        aid = album_ids_annotated[idx]
        if aid == album_id:
            continue
        row_data = lookup.loc[aid] if aid in lookup.index else {'album_name': None, 'artist_name': None}
        if input_artist and row_data['artist_name'] == input_artist:
            continue
        results.append({
            'album_id':    aid,
            'album_name':  row_data['album_name'],
            'artist_name': row_data['artist_name'],
            'distance':    round(dist, 4),
        })
        if len(results) == n:
            break

    return pd.DataFrame(results)

## Smoke test — Massive Attack

End-to-end validation of the full query pipeline. `search_artist("Massive Attack")` confirms the lookup table contains the expected artist and reveals the album IDs available to query. `recommend(album_id)` then runs a full KNN query and returns a labelled DataFrame.

This is the primary manual QA step: if the returned albums are plausible genre neighbours (trip-hop, electronic, ambient), the features and distance metric are working correctly. If results look random or obviously wrong, it indicates a problem upstream in feature construction.

In [ ]:
# Smoke test — find Massive Attack albums and recommend from one
results = search_artist("Massive Attack")
print(results.to_string())
print()

album_id = results.index[0]
recommend(album_id)